In [ ]:
import pyproj
pyproj.datadir.set_data_dir("/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/proj/9.2.0/share/proj")

In [ ]:
from eval import *
from utils_cartopy import global_mean
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from eval_shakirova import (
    ALBEDO_KERNEL_PATH,
    WATER_VAPOR_KERNEL_PATH,
    albedo_kernel_components,
    compute_responses,
    select_kernel_month,
)
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

In [ ]:
def cyclic_data(data, lon):
    values = data.to_numpy() if hasattr(data, 'to_numpy') else np.asarray(data)
    return add_cyclic_point(values, coord=np.asarray(lon), axis=-1)


# Figure 1

In [ ]:
output_dir = Path('Figures2/Figure1')
output_dir.mkdir(exist_ok=True, parents=True)
checkpoint_path = Path('checkpoints/fal/2011-2013_3,6,9,12_sob_fal_clearsky/best_model.pt')
config_path = Path('configs/model/fal/2011-2013_3,6,9,12_sob_fal_clearsky.yaml')
config = load_config(config_path)
vc = variable_config_from_omegaconf(config)
model, preprocessor, _ = load_model_and_preprocessor(config, checkpoint_path)

In [ ]:
years = list(range(1990, 2010+1))
raw_era5_paths = generate_paths_yearly(
    config.dataset.era5.raw_path, years, make_era5_filename
)

In [ ]:
raw_dataset = xr.open_mfdataset(raw_era5_paths, combine="nested", concat_dim="date")
ds = preprocessor.transform(raw_dataset)
ds_clear = preprocessor.transform(vc.clear_sky_input(raw_dataset))

In [ ]:
lat, lon                         = ds.latitude.values, ds.longitude.values
target, target_clear             =  vc.target_var, vc.clear_sky_target
target_label, target_label_clear = target.upper(), target_clear.upper()

pred = nn_pred(ds, model, preprocessor, vc, ["date", "latitude", "longitude"])
pred_clear = nn_pred(ds_clear, model, preprocessor, vc, ["date", "latitude", "longitude"], clear=True)
true = preprocessor.preprocessors[-1].inverse_transform(ds)[target]
true_clear = preprocessor.preprocessors[-1].inverse_transform(ds)[target_clear]

tsr_true, tsr_true_clear = true / SECONDS_PER_DAY, true_clear / SECONDS_PER_DAY
tsr_pred, tsr_pred_clear = pred / SECONDS_PER_DAY, pred_clear / SECONDS_PER_DAY

In [ ]:
def compute_metrics(true, pred):
    diff = pred - true
    mbe = np.mean(diff, axis=0)
    global_mean_mbe = float(global_mean(mbe))

    rmse = np.sqrt(np.mean(diff**2, axis=0))
    global_mean_rmse = float(global_mean(rmse))

    return mbe, global_mean_mbe, rmse, global_mean_rmse


In [ ]:
    
mbe, mbe_ann, rmse, rmse_ann = compute_metrics(tsr_true, tsr_pred)
mbe_clear, mbe_ann_clear, rmse_clear, rmse_ann_clear = compute_metrics(tsr_true_clear, tsr_pred_clear)

print(float(np.max(np.abs(rmse_clear))))


In [ ]:

def plot_contours_on_ax(ax, data, cmap, vmin, vmax, step, colorbar, colorbar_label=None):
    im = ax.contourf(
        lon, 
        lat, 
        data, 
        levels=np.arange(vmin, vmax+step, step), 
        transform=ccrs.PlateCarree(), 
        cmap=cmap, 
        vmin=vmin, 
        vmax=vmax
    )
    if colorbar:
        fig.colorbar(im, ax=ax, label=colorbar_label)

def plot_text_on_ax(ax, text, units):
    t = f"{text:0.2f}"
    ax.text(
        1.02,
        1,
        t + ' ' + units,
        transform=ax.transAxes,
        horizontalalignment="right",
        verticalalignment="bottom",
    )

def add_gridlines_on_ax(ax, i, j, n_rows):
    gl = ax.gridlines(draw_labels=True)
    
    # Left column only (j == 0)
    gl.left_labels = (j == 0)
    gl.right_labels = False
    
    # Bottom row only (i == len(axes) - 1)
    gl.bottom_labels = (i == n_rows - 1)
    gl.top_labels = False

In [ ]:
print([float(np.max(np.abs(f))) for f in [mbe, rmse, mbe_clear, rmse_clear]])

In [ ]:
fig, ax = plt.subplots(4, 2, figsize=(12,12), subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)), layout='constrained')
titles = [
    [r'$\bf{(a)}$ TSR (ERA5)', r'$\bf{(b)}$ TSRC (ERA5)'],
    [r'$\bf{(c)}$ TSR (NN)'  , r'$\bf{(d)}$ TSRC (NN)'],
    [r'$\bf{(e)}$ MBE'       , r'$\bf{(f)}$ MBE'],
    [r'$\bf{(g)}$ RMSE'      , r'$\bf{(h)}$ RMSE'],
]
annotations = [
    [float(global_mean(tsr_true).mean()), float(global_mean(tsr_true_clear).mean())],
    [float(global_mean(tsr_pred).mean()), float(global_mean(tsr_pred_clear).mean())],
    [mbe_ann, mbe_ann_clear],
    [rmse_ann, rmse_ann_clear],
]

plot_contours_on_ax(ax[0][0], tsr_true.mean('date')      , vmin=0  , vmax=400, step=20, cmap='Reds'  , colorbar=False)
plot_contours_on_ax(ax[1][0], tsr_pred.mean('date')      , vmin=0  , vmax=400, step=20, cmap='Reds'  , colorbar=False)
plot_contours_on_ax(ax[2][0], mbe                        , vmin=-20, vmax=20 , step=2 , cmap='RdBu_r', colorbar=False)
plot_contours_on_ax(ax[3][0], rmse                       , vmin=0  , vmax=20 , step=2 , cmap='Blues' , colorbar=False)
plot_contours_on_ax(ax[0][1], tsr_true_clear.mean('date'), vmin=0  , vmax=400, step=20, cmap='Reds'  , colorbar=True, colorbar_label=r"W m$^{-2}$")
plot_contours_on_ax(ax[1][1], tsr_pred_clear.mean('date'), vmin=0  , vmax=400, step=20, cmap='Reds'  , colorbar=True, colorbar_label=r"W m$^{-2}$")
plot_contours_on_ax(ax[2][1], mbe_clear                  , vmin=-20, vmax=20 , step=2 , cmap='RdBu_r', colorbar=True, colorbar_label=r"W m$^{-2}$")
plot_contours_on_ax(ax[3][1], rmse_clear                 , vmin=0  , vmax=20 , step=2 , cmap='Blues' , colorbar=True, colorbar_label=r"W m$^{-2}$")

for i in range(len(ax)):
    for j in range(len(ax[0])):
        ax[i][j].coastlines()
        ax[i][j].gridlines()
        ax[i][j].set_title(titles[i][j], loc='left')
        plot_text_on_ax(ax[i][j], annotations[i][j], 'W m$^{-2}$')
        add_gridlines_on_ax(ax[i][j], i, j, len(ax))

In [ ]:
fig.savefig(output_dir / 'fig1.png')

# Figure 2

In [ ]:
output_dir = Path('Figures2/Figure2')
output_dir.mkdir(exist_ok=True, parents=True)
years = [2015]
raw_era5_paths = generate_paths_yearly(config.dataset.era5.raw_path, years, make_era5_filename)
date = '2015-09'
raw = xr.open_mfdataset(raw_era5_paths, combine="nested", concat_dim="date").sel(date=date)

In [ ]:
kernel_paths = generate_paths_yearly(
    config.dataset.kernels.raw_path,
    years,
    lambda year: make_kernel_filename(year, 'fal'),
)
kernels_dataset = xr.open_mfdataset(
    kernel_paths, combine="nested", concat_dim="date"
)
true_kernel = kernels_dataset.sel(date=date)
kern_lat, kern_lon = true_kernel.latitude, true_kernel.longitude


In [ ]:
kernel_name = 'fal'
title = kernel_title(kernel_name)
label = kernel_label(kernel_name)

nn_kern_cld, grad_lon, grad_lat = compute_nn_kernel_autograd(raw, preprocessor, model, vc, kernel_name)
nn_kern_clr, _, _               = compute_nn_kernel_autograd(raw, preprocessor, model, vc, kernel_name, clear=True)

nn_kern_cld = xr.DataArray(
    data=nn_kern_cld,
    dims=["latitude", "longitude"],
    coords={"latitude": grad_lat, "longitude": grad_lon},
).interp(latitude=kern_lat, longitude=kern_lon, kwargs={"fill_value": "extrapolate"})
nn_kern_clr = xr.DataArray(
    data=nn_kern_clr,
    dims=["latitude", "longitude"],
    coords={"latitude": grad_lat, "longitude": grad_lon},
).interp(latitude=kern_lat, longitude=kern_lon, kwargs={"fill_value": "extrapolate"})

rrtm_kern_cld = true_kernel[kernel_name].sel(all_clr='all') * kernel_delta(kernel_name)
rrtm_kern_clr = true_kernel[kernel_name].sel(all_clr='clr') * kernel_delta(kernel_name)

In [ ]:
def compute_metrics(true, pred):
    diff = pred - true
    mbe = global_mean(diff)
    rmse = np.sqrt(global_mean(diff**2))

    return diff, mbe, rmse

In [ ]:
diff, mbe, rmse = compute_metrics(rrtm_kern_cld, nn_kern_cld)
diff_clear, mbe_clear, rmse_clear = compute_metrics(rrtm_kern_clr, nn_kern_clr)

In [ ]:
def plot_contours_on_ax(ax, data, cmap, vmin, vmax, step, colorbar, colorbar_label=None):
    im = ax.contourf(
        kern_lon, 
        kern_lat, 
        data, 
        levels=np.arange(vmin, vmax+step, step), 
        transform=ccrs.PlateCarree(), 
        cmap=cmap, 
        vmin=vmin, 
        vmax=vmax
    )
    if colorbar:
        fig.colorbar(im, ax=ax, label=colorbar_label)

def plot_text_on_ax(ax, text):
    ax.text(
        1.02,
        1,
        text,
        transform=ax.transAxes,
        horizontalalignment="right",
        verticalalignment="bottom",
    )

def add_gridlines_on_ax(ax, i, j, n_rows):
    gl = ax.gridlines(draw_labels=True)
    
    # Left column only (j == 0)
    gl.left_labels = (j == 0)
    gl.right_labels = False
    
    # Bottom row only (i == len(axes) - 1)
    gl.bottom_labels = (i == n_rows - 1)
    gl.top_labels = False

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(12,9), subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)), layout='constrained')
titles = [
    [r'$\bf{(a)}$ RRTM', r'$\bf{(b)}$ RRTM'],
    [r'$\bf{(c)}$ NN'  , r'$\bf{(d)}$ NN'],
    [r'$\bf{(e)}$ NN-RRTM' , r'$\bf{(f)}$ NN-RRTM'],
]
annotations = [
    [f"{float(global_mean(rrtm_kern_cld)):.2f} W m$^{{-2}}$ %$^{{-1}}$", f"{float(global_mean(rrtm_kern_clr)):.2f} W m$^{{-2}}$ %$^{{-1}}$"],
    [f"{float(global_mean(nn_kern_cld)):.2f} W m$^{{-2}}$ %$^{{-1}}$", f"{float(global_mean(nn_kern_clr)):.2f} W m$^{{-2}}$ %$^{{-1}}$"],
    [
        rf"MBE: {float(mbe):.2f} W m$^{{-2}}$ %$^{{-1}}$, RMSE: {float(rmse):.2f} W m$^{{-2}}$ %$^{{-1}}$",
        rf"MBE: {float(mbe_clear):.2f} W m$^{{-2}}$ %$^{{-1}}$, RMSE: {float(rmse_clear):.2f} W m$^{{-2}}$ %$^{{-1}}$",
    ],
]

plot_contours_on_ax(ax[0][0], rrtm_kern_cld[0]       , vmin=-4  , vmax=0,   step=0.25, cmap='Blues_r', colorbar=False)
plot_contours_on_ax(ax[1][0], nn_kern_cld            , vmin=-4  , vmax=0,   step=0.25, cmap='Blues_r', colorbar=False)
plot_contours_on_ax(ax[2][0], diff.mean('date')      , vmin=-0.5, vmax=0.5, step=0.05, cmap='RdBu_r', colorbar=False)
plot_contours_on_ax(ax[0][1], rrtm_kern_clr[0]       , vmin=-4  , vmax=0,   step=0.25, cmap='Blues_r', colorbar=True, colorbar_label="W m$^{-2}$ %$^{-1}$")
plot_contours_on_ax(ax[1][1], nn_kern_clr            , vmin=-4  , vmax=0,   step=0.25, cmap='Blues_r', colorbar=True, colorbar_label="W m$^{-2}$ %$^{-1}$")
plot_contours_on_ax(ax[2][1], diff_clear.mean('date'), vmin=-0.5, vmax=0.5, step=0.05, cmap='RdBu_r', colorbar=True, colorbar_label="W m$^{-2}$ %$^{-1}$")

for i in range(len(ax)):
    for j in range(len(ax[0])):
        ax[i][j].coastlines()
        ax[i][j].gridlines()
        ax[i][j].set_title(titles[i][j], loc='left')
        plot_text_on_ax(ax[i][j], annotations[i][j])
        add_gridlines_on_ax(ax[i][j], i, j, len(ax))

In [ ]:
fig.savefig(output_dir / 'fig2.png')

# Figure 3

In [ ]:
output_dir.mkdir(exist_ok=True, parents=True)
years = [2012, 2013]
raw_era5_paths = generate_paths_yearly(config.dataset.era5.raw_path, years, make_era5_filename)
raw = xr.open_mfdataset(raw_era5_paths, combine="nested", concat_dim="date")

In [ ]:
kernel_paths = generate_paths_yearly(
    config.dataset.kernels.raw_path,
    years,
    lambda year: make_kernel_filename(year, 'fal'),
)
kernels_dataset = xr.open_mfdataset(
    kernel_paths, combine="nested", concat_dim="date"
)

In [ ]:
base_date = '2020-09'
perturbed_date = '2025-09'
kernel_name = 'fal'
output_dir = Path('Figures2/Figure3')
output_dir.mkdir(exist_ok=True, parents=True)

base = select_date_state(raw, base_date)
perturbed = select_date_state(raw, perturbed_date)

nn_kern_1, nn_lon, nn_lat = compute_nn_kernel_autograd(
    perturbed.expand_dims(date=[perturbed_date]), preprocessor, model, vc, var=kernel_name
)
nn_kern_0, _, _ = compute_nn_kernel_autograd(
    base.expand_dims(date=[base_date]), preprocessor, model, vc, var=kernel_name
)

rrtm_lat, rrtm_lon = kernels_dataset.latitude, kernels_dataset.longitude
rrtm_kern_1 = kernels_dataset[kernel_name].sel(all_clr='all', date=perturbed_date).as_numpy()[0] * kernel_delta(kernel_name)
rrtm_kern_0 = kernels_dataset[kernel_name].sel(all_clr='all', date=base_date).as_numpy()[0] * kernel_delta(kernel_name)

if nn_kern_0.shape != rrtm_kern_0.shape:
    nn_kern_0 = interpolate_spatial_field(nn_kern_0, nn_lon, nn_lat, rrtm_lon, rrtm_lat)
    nn_kern_1 = interpolate_spatial_field(nn_kern_1, nn_lon, nn_lat, rrtm_lon, rrtm_lat)
    nn_lon, nn_lat = rrtm_lon, rrtm_lat

rrtm_diff = rrtm_kern_1 - rrtm_kern_0
nn_diff = nn_kern_1 - nn_kern_0
bias = nn_diff - rrtm_diff


In [ ]:
def cyclic_data(data, lon):
    values = data.to_numpy() if hasattr(data, 'to_numpy') else np.asarray(data)
    return add_cyclic_point(values, coord=np.asarray(lon), axis=-1)

def as_dataarray(data, lon, lat):
    if hasattr(data, 'dims'):
        return data
    return xr.DataArray(
        data,
        dims=['latitude', 'longitude'],
        coords={'latitude': lat, 'longitude': lon},
    )

def mean_units_text(data, lon, lat):
    field = as_dataarray(data, lon, lat)
    return f"{float(global_mean(field)):.2f} W m$^{{-2}}$ %$^{{-1}}$"

def residual_text(data, lon, lat):
    field = as_dataarray(data, lon, lat)
    mbe = float(global_mean(field))
    rmse = float(np.sqrt(global_mean(field**2)))
    return f"MBE: {mbe:.2f} W m$^{{-2}}$ %$^{{-1}}$, RMSE: {rmse:.2f} W m$^{{-2}}$ %$^{{-1}}$"

def plot_np_contours(ax, data, lon, lat, title, vmin, vmax, step, cmap='RdBu_r', title_padding=None):
    plot_data, plot_lon = cyclic_data(data, lon)
    im = ax.contourf(
        plot_lon,
        lat,
        plot_data,
        levels=np.arange(vmin, vmax+step, step),
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.coastlines()
    ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"rotation": 0}
    gl.ylabel_style = {"rotation": 0}
    ax.set_title(title, loc='left', pad=title_padding)
    return im



In [ ]:
fig, ax = plt.subplots(
    3, 3,
    figsize=(12,12),
    subplot_kw=dict(projection=ccrs.NorthPolarStereo(central_longitude=0)),
    layout='constrained',
    dpi=100
)
fig.get_layout_engine().set(rect=[0, 0, 0.92, 1])

kernel_vmin, kernel_vmax, kernel_step = -2, 0, 0.1
diff_vmin, diff_vmax, diff_step = -0.6, 0.6, 0.05
text = [
    [
        mean_units_text(rrtm_kern_1, rrtm_lon, rrtm_lat),
        mean_units_text(rrtm_kern_0, rrtm_lon, rrtm_lat),
        mean_units_text(rrtm_diff, rrtm_lon, rrtm_lat),
    ],
    [
        mean_units_text(nn_kern_1, nn_lon, nn_lat),
        mean_units_text(nn_kern_0, nn_lon, nn_lat),
        mean_units_text(nn_diff, nn_lon, nn_lat),
    ],
    [None, None, residual_text(bias, nn_lon, nn_lat)],
]

ims = []
ims.append(plot_np_contours(ax[0][0], rrtm_kern_1, rrtm_lon, rrtm_lat, rf'$\bf{{(a)}}$ RRTM {perturbed_date}', kernel_vmin, kernel_vmax, kernel_step, 'Blues_r'))
ims.append(plot_np_contours(ax[0][1], rrtm_kern_0, rrtm_lon, rrtm_lat, rf'$\bf{{(b)}}$ RRTM {base_date}', kernel_vmin, kernel_vmax, kernel_step, 'Blues_r'))
ims.append(plot_np_contours(ax[0][2], rrtm_diff, rrtm_lon, rrtm_lat, rf'$\bf{{(c)}}$ (a) - (b)', diff_vmin, diff_vmax, diff_step))
ims.append(plot_np_contours(ax[1][0], nn_kern_1, nn_lon, nn_lat, rf'$\bf{{(d)}}$ NN {perturbed_date}', kernel_vmin, kernel_vmax, kernel_step, 'Blues_r'))
ims.append(plot_np_contours(ax[1][1], nn_kern_0, nn_lon, nn_lat, rf'$\bf{{(e)}}$ NN {base_date}', kernel_vmin, kernel_vmax, kernel_step, 'Blues_r'))
ims.append(plot_np_contours(ax[1][2], nn_diff, nn_lon, nn_lat, rf'$\bf{{(f)}}$ (d) - (e)', diff_vmin, diff_vmax, diff_step))
ax[2][0].set_axis_off()
ax[2][1].set_axis_off()
ims.append(plot_np_contours(ax[2][2], bias, nn_lon, nn_lat, rf'$\bf{{(g)}}$ (f) - (c)', diff_vmin, diff_vmax, diff_step, title_padding=20))

for i in range(len(ax)):
    for j in range(len(ax[0])):
        if text[i][j] is not None:
            plot_text_on_ax(ax[i][j], text[i][j])
            
fig.canvas.draw()

left0 = ax[0][0].get_position()
left1 = ax[1][0].get_position()
right0 = ax[0][1].get_position()
right1 = ax[1][1].get_position()
x0 = min(left0.x0, left1.x0)
x1 = max(right0.x1, right1.x1)
y0 = min(left0.y0, left1.y0) - 0.055
kernel_cax = fig.add_axes([x0, y0, x1 - x0, 0.018])
kernel_cax.set_in_layout(False)
fig.colorbar(ims[1], cax=kernel_cax, orientation='horizontal', label="W m$^{-2}$ %$^{-1}$")

top = ax[0][2].get_position()
mid = ax[1][2].get_position()
bottom = ax[2][2].get_position()
y0 = min(top.y0, mid.y0, bottom.y0)
y1 = max(top.y1, mid.y1, bottom.y1)
diff_cax = fig.add_axes([top.x1 + 0.012, y0, 0.018, y1 - y0])
diff_cax.set_in_layout(False)
fig.colorbar(ims[-1], cax=diff_cax, label="W m$^{-2}$ %$^{-1}$")

# freeze the layout so savefig's internal re-draw can't shift anything again
fig.set_layout_engine('none')
fig.savefig(output_dir / 'fig3_second_order.png')

In [ ]:
processed_base = preprocessor.transform(base)
processed_perturbed = preprocessor.transform(perturbed)
delta_fal = perturbed.fal - base.fal
delta_ecod = processed_perturbed.ecod - processed_base.ecod

hybrid_diffs = []
for label_name, fields in (
    ('albedo only', ['fal']),
    ('cloud only', ['tcc', 'tclw', 'tciw', 'hcc', 'mcc', 'lcc']),
):
    available = [field for field in fields if field in perturbed]
    hybrid = base.assign({field: perturbed[field] for field in available})
    hybrid_kern, hybrid_lon, hybrid_lat = compute_nn_kernel_autograd(
        hybrid.expand_dims(date=[base_date]), preprocessor, model, vc, var=kernel_name
    )
    if hybrid_kern.shape != nn_kern_1.shape:
        hybrid_kern = interpolate_spatial_field(hybrid_kern, hybrid_lon, hybrid_lat, nn_lon, nn_lat)
        hybrid_lon, hybrid_lat = nn_lon, nn_lat
    hybrid_diffs.append((label_name, hybrid_kern - nn_kern_0, hybrid_lon, hybrid_lat))


In [ ]:
fig, ax = plt.subplots(
    2, 2,
    figsize=(12, 14),
    subplot_kw=dict(projection=ccrs.NorthPolarStereo(central_longitude=0)),
    layout='constrained',
)

im_fal = plot_np_contours(ax[0][0], delta_fal, delta_fal.longitude, delta_fal.latitude,
                           rf'$\bf{{(a)}}$ $\Delta$ albedo', -0.8, 0.8, 0.1)
im_nn_fal = plot_np_contours(ax[0][1], hybrid_diffs[0][1], hybrid_diffs[0][2], hybrid_diffs[0][3],
                              rf'$\bf{{(b)}}$ $\Delta K_a$', -0.6, 0.6, 0.05)
im_ecod = plot_np_contours(ax[1][0], delta_ecod, delta_ecod.longitude, delta_ecod.latitude,
                            rf'$\bf{{(c)}}$ $\Delta$ ecod', -1, 1, 0.1)
im_nn_ecod = plot_np_contours(ax[1][1], hybrid_diffs[1][1], hybrid_diffs[1][2], hybrid_diffs[1][3],
                               rf'$\bf{{(d)}}$ $\Delta K_c$', -0.6, 0.6, 0.05)

cbar_kwargs = dict(orientation='vertical', location='right', fraction=0.05, pad=0.04, aspect=25)

# left column: two separate colorbars, one per row
fig.colorbar(im_fal,  ax=ax[0][0], label='', **cbar_kwargs)
fig.colorbar(im_ecod, ax=ax[1][0], label='', **cbar_kwargs)

# right column: one colorbar shared across both rows
fig.colorbar(im_nn_ecod, ax=[ax[0][1], ax[1][1]], label="W m$^{-2}$ %$^{-1}$", **cbar_kwargs)

fig.savefig(output_dir / 'fig4_hybrid_second_order.png')

# Figure 4

In [ ]:
def zero_like_response(field):
    return xr.zeros_like(field)

def add_fields(fields):
    total = fields[0]
    for field in fields[1:]:
        total = total + field
    return total

def shakirova_terms(responses, sky):
    suffix = 'all' if sky == 'all' else 'clr'
    target = responses[f'dR_era5_{suffix}']
    if sky == 'all':
        kernel_parts = [responses['dR_a_k_all'], responses['dR_q_k_all'], responses['dR_c_k_all']]
        nn_parts = [responses['dR_a_nn_all'], responses['dR_q_nn_all'], responses['dR_c_nn_all']]
        nn_allcross = responses['dR_nn_all']
    else:
        kernel_parts = [responses['dR_a_k_clr'], responses['dR_q_k_clr']]
        nn_parts = [responses['dR_a_nn_clr'], responses['dR_q_nn_clr']]
        nn_allcross = responses['dR_nn_clr']

    kernel_sum = add_fields(kernel_parts)
    nn_sum = add_fields(nn_parts)
    return {
        'target': target,
        'kernel': [*kernel_parts, kernel_sum, target - kernel_sum],
        'nn': [*nn_parts, nn_sum, target - nn_sum],
        'allcross': [nn_allcross, target - nn_allcross],
    }


In [ ]:
def plot_shakirova_field(ax, field, title='', vmin=-60, vmax=60, step=5, cmap='RdBu_r', left_labels=False, bottom_labels=False, arctic=False):
    plot_data, plot_lon = cyclic_data(field, field.longitude)
    im = ax.contourf(
        plot_lon,
        field.latitude,
        plot_data,
        levels=np.arange(vmin, vmax + step, step),
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extend='both',
    )
    ax.coastlines()
    if arctic:
        ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = left_labels
    gl.bottom_labels = bottom_labels
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    if title:
        ax.set_title(title, loc='left')
    return im


def arctic_field(field):
    return field.where(field.latitude >= 60, drop=True)


def response_mean(field, arctic=False):
    return float(global_mean(arctic_field(field) if arctic else field))


def annotate_response(ax, field, residual=False, arctic=False):
    field = arctic_field(field) if arctic else field
    if residual:
        mbe = float(global_mean(field))
        rmse = float(np.sqrt(global_mean(field**2)))
        plot_text_on_ax(ax, f'MBE: {mbe:.2f} W m$^{{-2}}$, RMSE: {rmse:.2f} W m$^{{-2}}$')
    else:
        plot_text_on_ax(ax, f'{response_mean(field):.2f} W m$^{{-2}}$')


def panel_title(label, title):
    return rf'$\bf{{({label})}}$ {title}'


def add_bottom_colorbar(fig, im, axes_list, label, gap=0.018, height=0.014):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bboxes = [axis.get_tightbbox(renderer).transformed(fig.transFigure.inverted()) for axis in axes_list]
    x0 = min(axis.get_position().x0 for axis in axes_list)
    x1 = max(axis.get_position().x1 for axis in axes_list)
    y0 = min(bbox.y0 for bbox in bboxes) - gap - height
    cax = fig.add_axes([x0, y0, x1 - x0, height])
    cax.set_in_layout(False)
    return fig.colorbar(im, cax=cax, orientation='horizontal', label=label)


def make_shakirova_total_figure(sky, arctic=False):
    terms = shakirova_terms(shakirova_responses, sky)
    rows = [
        (terms['target'], None),
        (terms['kernel'][-2], terms['kernel'][-1]),
        (terms['nn'][-2], terms['nn'][-1]),
        (terms['allcross'][0], terms['allcross'][1]),
    ]

    figsize = (12, 24) if arctic else (12, 14)
    fig, ax = plt.subplots(
        4, 2,
        figsize=figsize,
        subplot_kw=dict(projection=ccrs.NorthPolarStereo(central_longitude=0) if arctic else ccrs.PlateCarree(central_longitude=180)),
        layout='constrained',
    )
    fig.get_layout_engine().set(rect=[0.08, 0.08, 1, 0.92], h_pad=0.04, hspace=0.07)

    response_im = residual_im = None
    response_titles = [r'$\Delta R$', r'$\Delta R_{sum}^{K}$', r'$\Delta R_{sum}^{NN}$', r'$\Delta R_{allcross}^{NN}$']
    residual_titles = ['', r'$R - \Delta R_{sum}^{K}$', r'$R - \Delta R_{sum}^{NN}$', r'$R - \Delta R_{allcross}^{NN}$']
    labels = iter('abcdefg')
    for row, (response, residual) in enumerate(rows):
        response_im = plot_shakirova_field(ax[row][0], response, panel_title(next(labels), response_titles[row]), vmin=-60, vmax=60, step=5, left_labels=True, bottom_labels=(row == 3), arctic=arctic)
        annotate_response(ax[row][0], response, arctic=arctic)
        if residual is None:
            ax[row][1].set_visible(False)
            continue
        residual_im = plot_shakirova_field(ax[row][1], residual, panel_title(next(labels), residual_titles[row]), vmin=-8, vmax=8, step=1, bottom_labels=(row == 3), arctic=arctic)
        annotate_response(ax[row][1], residual, residual=True, arctic=arctic)

    fig.canvas.draw()
    add_bottom_colorbar(fig, response_im, [ax[row][0] for row in range(4)], 'W m$^{-2}$')
    add_bottom_colorbar(fig, residual_im, [ax[row][1] for row in range(1, 4)], 'W m$^{-2}$')
    fig.set_layout_engine('none')
    region = '_arctic' if arctic else ''
    fig.savefig(shakirova_output_dir / f'shakirova_{sky}_total{region}.png', bbox_inches='tight')
    return fig


def make_shakirova_component_figure(sky, arctic=False):
    terms = shakirova_terms(shakirova_responses, sky)
    if sky == 'all':
        inputs = [
            (shakirova_anomaly.fal, r'$\Delta a$', 'a', -0.5, 0.5, 0.05, ''),
            (shakirova_anomaly.tcwv, r'$\Delta q$', 'q', -20, 20, 2, 'kg m$^{-2}$'),
            (shakirova_anomaly_ecod, r'$\Delta c$', 'c', -1, 1, 0.1, ''),
            (terms['target'], r'$\Delta R$', 'sum', -60, 60, 5, 'W m$^{-2}$'),
        ]
    else:
        inputs = [
            (shakirova_anomaly.fal, r'$\Delta a$', 'a', -0.5, 0.5, 0.05, ''),
            (shakirova_anomaly.tcwv, r'$\Delta q$', 'q', -20, 20, 2, 'kg m$^{-2}$'),
            (terms['target'], r'$\Delta R$', 'sum', -60, 60, 5, 'W m$^{-2}$'),
        ]

    figsize = (12, 4 * len(inputs)) if arctic else (12, 2.5 * len(inputs))
    fig, ax = plt.subplots(
        len(inputs), 3,
        figsize=figsize,
        subplot_kw=dict(projection=ccrs.NorthPolarStereo(central_longitude=0) if arctic else ccrs.PlateCarree(central_longitude=180)),
        layout='constrained',
    )
    fig.get_layout_engine().set(rect=[0.08, 0.09, 1, 0.92], h_pad=0.04, hspace=0.07)

    response_im = None
    panel_labels = [
        ['a', 'e', 'i'],
        ['b', 'f', 'j'],
        ['c', 'g', 'k'],
        ['d', 'h', 'l'],
    ]
    for row, (input_field, input_title, response_part, vmin, vmax, step, cbar_label) in enumerate(inputs):
        im = plot_shakirova_field(ax[row][0], input_field, panel_title(panel_labels[row][0], input_title), vmin=vmin, vmax=vmax, step=step, left_labels=True, bottom_labels=(row == len(inputs) - 1), arctic=arctic)
        plot_text_on_ax(ax[row][0], f'{response_mean(input_field, arctic=arctic):.2f}' + (f' {cbar_label}' if cbar_label else ''))
        fig.colorbar(im, ax=ax[row][0], orientation='vertical', label=cbar_label, shrink=0.7)

        for col, source in enumerate(['kernel', 'nn'], start=1):
            field = terms[source][row]
            title = rf'$\Delta R_{{{response_part}}}^{{{"K" if source == "kernel" else "NN"}}}$'
            response_im = plot_shakirova_field(ax[row][col], field, panel_title(panel_labels[row][col], title), vmin=-60, vmax=60, step=5, bottom_labels=(row == len(inputs) - 1), arctic=arctic)
            annotate_response(ax[row][col], field, arctic=arctic)

    fig.canvas.draw()
    add_bottom_colorbar(fig, response_im, [ax[row][col] for row in range(len(inputs)) for col in (1, 2)], 'W m$^{-2}$')
    fig.set_layout_engine('none')
    region = '_arctic' if arctic else ''
    fig.savefig(shakirova_output_dir / f'shakirova_{sky}_components{region}.png', bbox_inches='tight')
    return fig


In [ ]:
years = [1992, 2020]
raw_era5_paths = generate_paths_yearly(config.dataset.era5.raw_path, years, make_era5_filename)
raw = xr.open_mfdataset(raw_era5_paths, combine="nested", concat_dim="date")

model, preprocessor, _ = load_model_and_preprocessor(config, checkpoint_path, downscaling=False)

In [ ]:
shakirova_output_dir = Path('Figures2/Figure4')
shakirova_output_dir.mkdir(exist_ok=True, parents=True)
shakirova_base_date = '1992-09'
shakirova_perturbed_date = '2020-09'
response_save_path = shakirova_output_dir / f'saved_responses3.nc'

K_a = xr.open_dataset(ALBEDO_KERNEL_PATH)
shakirova_grid = {'latitude': K_a.latitude, 'longitude': K_a.longitude}
shakirova_ds = raw.interp(**shakirova_grid)
shakirova_base = select_date_state(shakirova_ds, shakirova_base_date)
shakirova_perturbed = select_date_state(shakirova_ds, shakirova_perturbed_date)
shakirova_anomaly = shakirova_perturbed - shakirova_base
shakirova_anomaly_ecod = (
    preprocessor.transform(shakirova_perturbed) - preprocessor.transform(shakirova_base)
).ecod.compute()

if response_save_path.exists():
    shakirova_responses = xr.load_dataset(response_save_path)
else:
    shakirova_responses = compute_responses(
        shakirova_perturbed,
        shakirova_base,
        model,
            preprocessor,
        vc,
        shakirova_base_date,
        shakirova_perturbed_date,
        Path(config.dataset.era5.raw_path),
    )
    shakirova_responses.to_netcdf(response_save_path)


In [ ]:
fig_shakirova_all_total = make_shakirova_total_figure('all')
fig_shakirova_clr_total = make_shakirova_total_figure('clr')
fig_shakirova_all_components = make_shakirova_component_figure('all')
fig_shakirova_clr_components = make_shakirova_component_figure('clr')
fig_shakirova_all_total_arctic = make_shakirova_total_figure('all', arctic=True)
fig_shakirova_clr_total_arctic = make_shakirova_total_figure('clr', arctic=True)
fig_shakirova_all_components_arctic = make_shakirova_component_figure('all', arctic=True)
fig_shakirova_clr_components_arctic = make_shakirova_component_figure('clr', arctic=True)


# Architecture Figure

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D

# ---- Layer definitions ----
input_labels = [
    "tisr", "fal", "sp", "hcc", "mcc", "lcc",
    "tciw", "tclw", "tcwv", "tco3", "ecod", r"ecod$\times$fal"
]
# True where the input is cloud-dependent (zeroed for clear-sky target)
cloud_mask = [
    False, False, False, True, True, True,
    True, True, False, False, True, True
]

n_in, n_hidden, n_out = 12, 11, 1

fig, ax = plt.subplots(figsize=(9, 7))

# ---- Node positions ----
x_in, x_hidden, x_out = 0.0, 3.0, 6.0

y_in = [n_in - 1 - i for i in range(n_in)]          # 11 down to 0
y_hidden_span = n_in - 1                             # match input span (0..11)
y_hidden = [y_hidden_span - i * (y_hidden_span / (n_hidden - 1)) for i in range(n_hidden)]
y_out = [sum(y_in) / len(y_in)]                      # centered output

node_r = 0.28

# ---- Draw edges first (so nodes sit on top) ----
for yi in y_in:
    for yh in y_hidden:
        ax.add_line(Line2D([x_in, x_hidden], [yi, yh],
                            color="gray", alpha=0.35, linewidth=0.6, zorder=1))

for yh in y_hidden:
    ax.add_line(Line2D([x_hidden, x_out], [yh, y_out[0]],
                        color="gray", alpha=0.35, linewidth=0.6, zorder=1))

# ---- Input nodes ----
for yi, label, is_cloud in zip(y_in, input_labels, cloud_mask):
    color = "#a8c8f0" if is_cloud else "white"
    ax.add_patch(Circle((x_in, yi), node_r, facecolor=color,
                         edgecolor="black", zorder=2))
    ax.text(x_in - 0.5, yi, label, ha="right", va="center", fontsize=9)

# ---- Hidden nodes ----
for yh in y_hidden:
    ax.add_patch(Circle((x_hidden, yh), node_r, facecolor="#e8e8e8",
                         edgecolor="black", zorder=2))
    #ax.text(x_hidden, yh, "tanh", ha="center", va="center", fontsize=6)

# ---- Output node ----
ax.add_patch(Circle((x_out, y_out[0]), node_r * 1.1, facecolor="#f5c98a",
                     edgecolor="black", zorder=2))
#ax.text(x_out, y_out[0], "tanh", ha="center", va="center", fontsize=6)
ax.text(x_out + 0.5, y_out[0], r"tsr(c)", ha="left", va="center", fontsize=10)

# ---- Legend: cloud-masking note ----
legend_y = -2.0
ax.add_patch(Circle((x_in, legend_y), 0.18, facecolor="#a8c8f0",
                     edgecolor="black", zorder=2))
ax.text(x_in + 0.4, legend_y,
        "Cloud-dependent input"
        "set to 0 when predicting clear-sky flux tsrc",
        ha="left", va="center", fontsize=8)

# ---- Layer titles ----
ax.text(x_in, n_in + 0.6, "Input\n(12)", ha="center", va="bottom", fontsize=10, weight="bold")
ax.text(x_hidden, n_in + 0.6, "Hidden\n(11)", ha="center", va="bottom", fontsize=10, weight="bold")
ax.text(x_out, n_in + 0.6, "Output\n(1)", ha="center", va="bottom", fontsize=10, weight="bold")

# ---- Cleanup ----
ax.set_xlim(-2.2, 8.0)
ax.set_ylim(legend_y - 1.2, n_in + 1.5)
ax.set_aspect("equal")
ax.axis("off")

plt.tight_layout()
plt.savefig("Figures/nn_diagram.png", dpi=300, bbox_inches="tight")
plt.savefig("Figures/nn_diagram.pdf", bbox_inches="tight")
print("Saved nn_diagram.png and nn_diagram.pdf")
